# CONSTRUCT-SAFE AI: Phase 2C GPU Training
This notebook automates the dataset acquisition, training, and evaluation of the YOLO11n custom PPE model using Google Colab's free GPU.

In [ ]:
!pip install ultralytics roboflow
from IPython import display
display.clear_output()
import ultralytics
ultralytics.checks()

## 1. Download Dataset
Using the Ultralytics Construction-PPE dataset.

In [ ]:
import os
import urllib.request
import zipfile

dataset_url = 'https://github.com/ultralytics/assets/releases/download/v0.0.0/construction-ppe.zip'
zip_path = 'construction-ppe.zip'
extract_dir = 'datasets/construction_safety'

if not os.path.exists(extract_dir):
    print('Downloading dataset...')
    urllib.request.urlretrieve(dataset_url, zip_path)
    print('Extracting...')
    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    os.remove(zip_path)
    print('Dataset ready.')
else:
    print('Dataset already exists.')

## 2. Prepare Config

In [ ]:
yaml_content = """path: /content/datasets/construction_safety
train: images/train
val: images/val
test: images/test

names:
  0: helmet
  1: gloves
  2: vest
  3: boots
  4: goggles
  5: none
  6: Person
  7: no_helmet
  8: no_goggle
  9: no_gloves
  10: no_boots
"""

with open('dataset.yaml', 'w') as f:
    f.write(yaml_content)

## 3. Train Model
Training on the full dataset using GPU.

In [ ]:
from ultralytics import YOLO

# Load a pretrained YOLO11n model
model = YOLO('yolo11n.pt')

# Train the model
results = model.train(
    data='dataset.yaml',
    epochs=50,       # Start with 50 epochs
    imgsz=640,
    batch=16,
    device=0,        # Use GPU 0
    patience=15,     # Early stopping
    project='runs',
    name='ppe_training_full',
    verbose=True
)

## 4. Evaluate on Untouched Test Set

In [ ]:
# Load the best weights
best_model = YOLO('runs/ppe_training_full/weights/best.pt')

# Evaluate on test split
metrics = best_model.val(data='dataset.yaml', split='test')

print("\n--- Overall Metrics ---")
print(f"mAP50-95: {metrics.box.map}")
print(f"mAP50: {metrics.box.map50}")

print("\n--- Per-Class Metrics ---")
print(f"{'-Class-':<15} {'-Precision-':<12} {'-Recall-':<12} {'-mAP50-':<12} {'-mAP50-95-':<12}")
for i, c in enumerate(metrics.box.ap_class_index):
    name = best_model.names[c]
    p = metrics.box.p[i]
    r = metrics.box.r[i]
    map50 = metrics.box.ap50[i]
    map_all = metrics.box.ap[i]
    print(f"{name:<15} {p:<12.4f} {r:<12.4f} {map50:<12.4f} {map_all:<12.4f}")

## 5. Visual Predictions (False Positives / Negatives)

In [ ]:
import glob
import random
from IPython.display import Image, display

test_images = glob.glob('/content/datasets/construction_safety/images/test/*.jpg')
samples = random.sample(test_images, 5)

for img_path in samples:
    results = best_model.predict(source=img_path, save=True, project='runs', name='predict')
    # Display original and predicted
    print(f"Predictions for {os.path.basename(img_path)}:")
    display(Image(filename=results[0].save_dir + '/' + os.path.basename(img_path)))